In [ ]:
import torch
import subprocess
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import PeftModel

# --- 1. CONFIGURATION ---
BASE_MODEL = "Qwen/Qwen2.5-Coder-1.5B"
ADAPTER_DIR = "./krakatau_lora_model"

print("Loading AI Brain...")
tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL)
base_model = AutoModelForCausalLM.from_pretrained(BASE_MODEL, device_map="auto")

# Snap the LoRA adapter onto the base model
model = PeftModel.from_pretrained(base_model, ADAPTER_DIR)
print("AI Ready!\n")

def ai_to_java(task_description):
    # --- 2. PROMPT THE AI ---
    # MUST match your training data format exactly!
    prompt = (
        f"Write Java bytecode in Krakatau Jasmin syntax for the following task.\n"
        f"Task: {task_description}\n"
        f"Code:\n"
    )
    
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    
    print(f"Thinking about: '{task_description}'...")
    outputs = model.generate(
        **inputs,
        max_new_tokens=500,     # Max length of bytecode
        temperature=0.1,        # Keep it low so it doesn't hallucinate syntax
        do_sample=True,
        eos_token_id=tokenizer.eos_token_id
    )
    
    # Extract only the newly generated code
    full_output = tokenizer.decode(outputs[0], skip_special_tokens=True)
    generated_code = full_output[len(prompt):].strip()
    
    print("\n================ AI BYTECODE ================")
    print(generated_code)
    print("=============================================\n")
    
    # --- 3. EXECUTE THE PIPELINE ---
    try:
        # Step A: Save AI output to a file
        with open("Solution.j", "w", encoding="utf-8") as f:
            f.write(generated_code)
            
        # Step B: Assemble with your Python 3 Krakatau setup
        print("Assembling with Krakatau...")
        subprocess.run(
            ["python", "./Krakatau/assemble.py", "-out", ".", "Solution.j"], 
            check=True
        )
        
        # Step C: Run the compiled Java class!
        print("Running Java Virtual Machine...\n")
        print(">>> OUTPUT:")
        subprocess.run(["java", "-cp", ".", "Solution"], check=True)
        print("\n>>> DONE.")
        
    except subprocess.CalledProcessError as e:
        print(f"\n[!] Pipeline Failed: The AI generated invalid bytecode or Krakatau threw an error.")

# --- 4. TEST IT ---
if __name__ == "__main__":
    # Put whatever English command you want here!
    my_prompt = "Write a function that prints the number 42 to the console."
    ai_to_java(my_prompt)

Loading AI Brain...


Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

[transformers] Setting `pad_token_id` to `eos_token_id`:151643 for open-end generation.


AI Ready!

Thinking about: 'Write a function that prints the number 42 to the console.'...

================ AI BYTECODE ================
.version 55 0 
.class super Solution 
.super java/lang/Object 

.method <init> : ()V 
    .code stack 1 locals 1 
L0:     aload_0 
L1:     invokespecial Method java/lang/Object <init> ()V 
L4:     return 
L5:     
        .linenumbertable 
            L0 4 
        .end linenumbertable 
    .end code 
.end method 

.method public print42 : ()V 
    .code stack 1 locals 2 
L0:     ldc 42 
L2:     invokevirtual Method java/io/PrintStream println (I)V 
L5:     return 
L6:     
        .linenumbertable 
            L0 7 
        .end linenumbertable 
    .end code 
.end method 
.sourcefile 'Solution.java' 
.end class

Assembling with Krakatau...

[!] Pipeline Failed: The AI generated invalid bytecode or Krakatau threw an error.
